In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load dataset
df = pd.read_csv("Exam_Score_Prediction.csv")

# Separate features and target
# exam_score is the value we want to predict
X = df.drop(["exam_score", "student_id"], axis=1)
y = df["exam_score"]

# Encode categorical columns
X = pd.get_dummies(X, dtype=int, drop_first=True)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Evaluate model
y_pred = model.predict(X_test_scaled)

print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

# Save trained model
with open("student_score_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save scaler
with open("student_score_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save feature columns
with open("student_score_columns.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)

print("Student Score Prediction model files saved successfully!")

MSE: 95.50209594902535
MAE: 7.863070298665213
RMSE: 9.772517380338872
R2 Score: 0.7330084088180806
Student Score Prediction model files saved successfully!


In [2]:

# Code Start

import pickle
import pandas as pd
import gradio as gr

# Load trained model files
with open("student_score_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("student_score_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("student_score_columns.pkl", "rb") as f:
    columns = pickle.load(f)

# Load dataset to get actual category options
df = pd.read_csv("Exam_Score_Prediction.csv")

# Get category values directly from the dataset
gender_options = sorted(df["gender"].dropna().astype(str).unique().tolist())
course_options = sorted(df["course"].dropna().astype(str).unique().tolist())
internet_options = sorted(df["internet_access"].dropna().astype(str).unique().tolist())
sleep_quality_options = sorted(df["sleep_quality"].dropna().astype(str).unique().tolist())
study_method_options = sorted(df["study_method"].dropna().astype(str).unique().tolist())
facility_options = sorted(df["facility_rating"].dropna().astype(str).unique().tolist())
difficulty_options = sorted(df["exam_difficulty"].dropna().astype(str).unique().tolist())


# Prediction function
def predict_score(
    age, gender, course, study_hours, class_attendance,
    internet_access, sleep_hours, sleep_quality,
    study_method, facility_rating, exam_difficulty
):
    # Prepare student input
    data = pd.DataFrame([{
        "age": age,
        "gender": gender,
        "course": course,
        "study_hours": study_hours,
        "class_attendance": class_attendance,
        "internet_access": internet_access,
        "sleep_hours": sleep_hours,
        "sleep_quality": sleep_quality,
        "study_method": study_method,
        "facility_rating": facility_rating,
        "exam_difficulty": exam_difficulty
    }])

    # Encode categorical features
    data = pd.get_dummies(data, dtype=int, drop_first=True)

    # Ensure features match the training data
    data = data.reindex(columns=columns, fill_value=0)

    # Scale features
    data_scaled = scaler.transform(data)

    # Predict exam score
    prediction = float(model.predict(data_scaled)[0])

    # Keep displayed score within 0–100
    prediction = max(0.0, min(100.0, prediction))

    return f"Predicted Exam Score: {prediction:.2f} / 100"


# Create Gradio interface
app = gr.Interface(
    fn=predict_score,
    inputs=[
        gr.Number(label="Age", value=20, minimum=1),
        gr.Dropdown(choices=gender_options, label="Gender"),
        gr.Dropdown(choices=course_options, label="Course"),
        gr.Number(label="Study Hours", value=4, minimum=0),
        gr.Number(
            label="Class Attendance (%)",
            value=80,
            minimum=0,
            maximum=100
        ),
        gr.Dropdown(choices=internet_options, label="Internet Access"),
        gr.Number(
            label="Sleep Hours",
            value=7,
            minimum=0,
            maximum=24
        ),
        gr.Dropdown(choices=sleep_quality_options, label="Sleep Quality"),
        gr.Dropdown(choices=study_method_options, label="Study Method"),
        gr.Dropdown(choices=facility_options, label="Facility Rating"),
        gr.Dropdown(choices=difficulty_options, label="Exam Difficulty")
    ],
    outputs=gr.Textbox(label="Predicted Exam Score"),
    title="Exam Score Prediction System",
    description=(
        "Enter student information, study habits, and exam details "
        "to predict the expected exam score."
    ),
    submit_btn="Predict Exam Score",
    clear_btn="Clear"
)

# Launch application
if __name__ == "__main__":
    app.launch()

# Code End


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import pandas as pd
import pickle
from flask import Flask, render_template_string, request

# Load dataset
df = pd.read_csv("Exam_Score_Prediction.csv")
df.columns = df.columns.str.strip()

# Load trained model files
with open("student_score_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("student_score_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("student_score_columns.pkl", "rb") as f:
    model_columns = pickle.load(f)

# Create Flask application
app = Flask(__name__)

# Dropdown options from dataset
fields = {
    "gender": sorted(df["gender"].dropna().astype(str).unique().tolist()),
    "course": sorted(df["course"].dropna().astype(str).unique().tolist()),
    "internet_access": sorted(df["internet_access"].dropna().astype(str).unique().tolist()),
    "sleep_quality": sorted(df["sleep_quality"].dropna().astype(str).unique().tolist()),
    "study_method": sorted(df["study_method"].dropna().astype(str).unique().tolist()),
    "facility_rating": sorted(df["facility_rating"].dropna().astype(str).unique().tolist()),
    "exam_difficulty": sorted(df["exam_difficulty"].dropna().astype(str).unique().tolist())
}

# HTML page
HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>Student Exam Score Prediction</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: #f2f5f9;
            margin: 0;
            padding: 30px;
        }
        .container {
            max-width: 750px;
            margin: auto;
            background: white;
            padding: 30px;
            border-radius: 12px;
            box-shadow: 0 3px 12px #cccccc;
        }
        h1 {
            text-align: center;
            color: #183b65;
        }
        .grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 18px;
        }
        label {
            display: block;
            margin-bottom: 6px;
            font-weight: bold;
        }
        input, select {
            width: 100%;
            padding: 10px;
            box-sizing: border-box;
            border: 1px solid #bbb;
            border-radius: 5px;
        }
        button {
            width: 100%;
            padding: 13px;
            margin-top: 25px;
            background: #1769aa;
            color: white;
            border: none;
            border-radius: 6px;
            font-size: 16px;
            cursor: pointer;
        }
        button:hover {
            background: #104e80;
        }
        .result {
            margin-top: 20px;
            padding: 15px;
            background: #e6f5e9;
            border-radius: 6px;
            text-align: center;
            color: #185c2b;
            font-size: 20px;
            font-weight: bold;
        }
        .error {
            color: #a11;
            background: #ffeded;
            padding: 12px;
            margin-top: 15px;
            border-radius: 5px;
        }
        @media (max-width: 600px) {
            .grid { grid-template-columns: 1fr; }
        }
    </style>
</head>
<body>
<div class="container">
    <h1>Student Exam Score Prediction</h1>

    <form method="POST" action="/">
        <div class="grid">
            <div>
                <label>Age</label>
                <input type="number" name="age" min="1" max="100"
                       value="{{ values.get('age', 20) }}" required>
            </div>

            <div>
                <label>Gender</label>
                <select name="gender" required>
                    {% for option in fields.gender %}
                    <option value="{{ option }}"
                    {% if values.get('gender', fields.gender[0]) == option %}selected{% endif %}>
                        {{ option }}
                    </option>
                    {% endfor %}
                </select>
            </div>

            <div>
                <label>Course</label>
                <select name="course" required>
                    {% for option in fields.course %}
                    <option value="{{ option }}"
                    {% if values.get('course', fields.course[0]) == option %}selected{% endif %}>
                        {{ option }}
                    </option>
                    {% endfor %}
                </select>
            </div>

            <div>
                <label>Study Hours Per Day</label>
                <input type="number" name="study_hours" min="0" max="24"
                       step="0.1" value="{{ values.get('study_hours', 4) }}" required>
            </div>

            <div>
                <label>Class Attendance (%)</label>
                <input type="number" name="class_attendance" min="0" max="100"
                       step="0.1" value="{{ values.get('class_attendance', 80) }}" required>
            </div>

            <div>
                <label>Internet Access</label>
                <select name="internet_access" required>
                    {% for option in fields.internet_access %}
                    <option value="{{ option }}"
                    {% if values.get('internet_access', fields.internet_access[0]) == option %}selected{% endif %}>
                        {{ option }}
                    </option>
                    {% endfor %}
                </select>
            </div>

            <div>
                <label>Sleep Hours</label>
                <input type="number" name="sleep_hours" min="0" max="24"
                       step="0.1" value="{{ values.get('sleep_hours', 7) }}" required>
            </div>

            <div>
                <label>Sleep Quality</label>
                <select name="sleep_quality" required>
                    {% for option in fields.sleep_quality %}
                    <option value="{{ option }}"
                    {% if values.get('sleep_quality', fields.sleep_quality[0]) == option %}selected{% endif %}>
                        {{ option }}
                    </option>
                    {% endfor %}
                </select>
            </div>

            <div>
                <label>Study Method</label>
                <select name="study_method" required>
                    {% for option in fields.study_method %}
                    <option value="{{ option }}"
                    {% if values.get('study_method', fields.study_method[0]) == option %}selected{% endif %}>
                        {{ option }}
                    </option>
                    {% endfor %}
                </select>
            </div>

            <div>
                <label>Facility Rating</label>
                <select name="facility_rating" required>
                    {% for option in fields.facility_rating %}
                    <option value="{{ option }}"
                    {% if values.get('facility_rating', fields.facility_rating[0]) == option %}selected{% endif %}>
                        {{ option }}
                    </option>
                    {% endfor %}
                </select>
            </div>

            <div>
                <label>Exam Difficulty</label>
                <select name="exam_difficulty" required>
                    {% for option in fields.exam_difficulty %}
                    <option value="{{ option }}"
                    {% if values.get('exam_difficulty', fields.exam_difficulty[0]) == option %}selected{% endif %}>
                        {{ option }}
                    </option>
                    {% endfor %}
                </select>
            </div>
        </div>

        <button type="submit">Predict Exam Score</button>
    </form>

    {% if prediction is not none %}
    <div class="result">
        Predicted Exam Score: {{ prediction }} / 100
    </div>
    {% endif %}

    {% if error %}
    <div class="error">{{ error }}</div>
    {% endif %}
</div>
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def home():
    prediction = None
    error = None
    values = {}

    if request.method == "POST":
        try:
            # Read values submitted by the form
            values = request.form.to_dict()

            # Build one input row
            input_data = {
                "age": float(values["age"]),
                "gender": values["gender"],
                "course": values["course"],
                "study_hours": float(values["study_hours"]),
                "class_attendance": float(values["class_attendance"]),
                "internet_access": values["internet_access"],
                "sleep_hours": float(values["sleep_hours"]),
                "sleep_quality": values["sleep_quality"],
                "study_method": values["study_method"],
                "facility_rating": values["facility_rating"],
                "exam_difficulty": values["exam_difficulty"]
            }

            input_df = pd.DataFrame([input_data])

            # Apply the same one-hot encoding as training
            input_df = pd.get_dummies(input_df, dtype=int, drop_first=True)

            # Match the exact training columns and order
            input_df = input_df.reindex(
                columns=model_columns,
                fill_value=0
            )

            # Scale input using the saved scaler
            scaled_input = scaler.transform(input_df)

            # Predict exam score
            raw_prediction = float(model.predict(scaled_input)[0])

            # Keep displayed score within the expected 0–100 range
            prediction = round(max(0, min(100, raw_prediction)), 2)

        except Exception as e:
            error = "Prediction error: " + str(e)

    return render_template_string(
        HTML,
        fields=fields,
        values=values,
        prediction=prediction,
        error=error
    )

print("Student Score Prediction App is starting...")
app.run(host="127.0.0.1", port=5004, debug=False, use_reloader=False)

# Code End


Student Score Prediction App is starting...
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5004
Press CTRL+C to quit
127.0.0.1 - - [11/Oct/2026 23:54:46] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [11/Oct/2026 23:54:46] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [11/Oct/2026 23:54:53] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [11/Oct/2026 23:55:16] "POST / HTTP/1.1" 200 -
